# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook:** c24-29 Optimizer Pipeline Workflow  
**Author:** Jasper Cluistra  
**Last Updated:** 2026-05-17

---

### Goal
Step-by-step walkthrough of the full optimizer pipeline for a single design candidate. Useful for debugging individual stages, inspecting intermediate outputs, and verifying that module changes behave correctly before running the full GA.

### Pipeline stages
| Stage | Module | Description |
|-------|--------|-------------|
| 1 | Setup | Load stock, search space, path configuration |
| 2 | Geometry | Generate random design, compute member topology |
| 3 | Feasibility | EC5 filter â€” eliminate infeasible slotÃ—stock pairs |
| 4 | Cost matrix | Compute COâ‚‚e assignment costs |
| 5 | MILP | Solve optimal stock assignment |
| 6 | GNN | Predict structural feasibility per member |
| 7 | Fitness | Compute normalised multi-objective fitness score |
| 8 | Export | Write vertices, edges, and assignment results to disk |

# 1. Setup
Resolve paths, load stock and search space. Set `MODEL_PREFIX` to a trained surrogate prefix to enable the GNN stage; leave as `None` to skip it.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate / "config.py").exists():
        REPO_ROOT = candidate
        break

SRC_PATH       = REPO_ROOT / "src"
WORKFLOWS_PATH = REPO_ROOT / "workflows"
for path in (REPO_ROOT, SRC_PATH, WORKFLOWS_PATH):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import config
import c00_headquarter_params as c11_params

MODEL_PREFIX = None  # e.g. "ID20260516_182257_LR1e-04_EP200_BS64_PW2.5_ROC0.863"

# ── Search space ──────────────────────────────────────────────────────
json_path = config.DATA_IO_PATH / f"search_space_{c11_params.GRID}.json"
if not json_path.exists():
    raise FileNotFoundError(
        f"search_space_{c11_params.GRID}.json not found. "
        "Run c12_15 to generate it."
    )
with open(json_path, "r") as f:
    optimizer_search_space = json.load(f)
print(f"Search space loaded: {len(optimizer_search_space)} variables")

# ── Stock ────────────────────────────────────────────────────────────
stock_path = config.TIMBER_STOCK_PATH / "complete_timber_v2.csv"
if not stock_path.exists():
    raise FileNotFoundError(f"Stock CSV not found: {stock_path}. Run c16 to generate it.")

df_input_stock = None
for opts in [
    {"sep": ",", "encoding": "utf-8"},
    {"sep": ";", "encoding": "utf-8"},
    {"sep": ",", "encoding": "latin1"},
    {"sep": ";", "encoding": "latin1"},
]:
    try:
        _df = pd.read_csv(stock_path, **opts)
        if _df.shape[1] > 1:
            df_input_stock = _df
            print(f"Stock loaded: {len(_df)} elements  (sep='{opts['sep']}', encoding='{opts['encoding']}')")
            break
    except Exception:
        pass

if df_input_stock is None:
    raise ValueError("Could not parse stock CSV with tested delimiter/encoding combinations.")

df_input_stock.columns = df_input_stock.columns.str.strip()
display(df_input_stock.head(3))

# 2. Geometry
Generate a random design candidate and compute the truss topology (nodes, members, lengths).

In [ ]:
import matplotlib.pyplot as plt
import config
import c24_stage_geometry as stage_geometry

geometry_out = stage_geometry.run_random_geometry_stage(
    json_path=json_path if "json_path" in globals() else None,
    optimizer_search_space=optimizer_search_space if "optimizer_search_space" in globals() else None,
    sample_id=0,
)

my_random_design = geometry_out["my_random_design"]
vertices_list = geometry_out["vertices_list"]
df_vertices = geometry_out["df_vertices"]
df_edges = geometry_out["df_edges"]
df_geometry_overview = geometry_out["df_geometry_overview"]

print(f"Geometry: {len(df_vertices)} nodes, {len(df_edges)} members")
print(
    f"Length range [m]: {df_geometry_overview['length_m'].min():.3f}"
    f" - {df_geometry_overview['length_m'].max():.3f}"
)
display(df_geometry_overview[["edge_id", "V1", "V2", "length_m"]].head(5))

df_vertices.to_csv(config.DATA_IO_PATH / "df_vertices.csv", index=False)
df_edges.to_csv(config.DATA_IO_PATH / "df_edges.csv", index=False)

fig, ax = stage_geometry.plot_geometry_preview(
    df_vertices=df_vertices,
    df_edges=df_edges,
    figsize=(8, 7),
)
plt.show()

# 3â€“7. Pipeline stages
Run the full pipeline on the geometry from section 2: feasibility filter â†’ cost matrix â†’ MILP â†’ GNN â†’ fitness.

In [ ]:
import importlib
import json
import pandas as pd
import config
from workflows import c26_stage_cost_matrix as stage_cost
from workflows import c27_stage_MILP as stage_milp

importlib.reload(stage_cost)
importlib.reload(stage_milp)

import numpy as np
from workflows import c25_stage_feasibility as stage_feas

verts = df_vertices[df_vertices['sample_id'] == 0].copy()
verts['v_idx'] = verts['vertex_index'].str.replace('v', '').astype(int)
verts = verts.sort_values('v_idx').reset_index(drop=True)
node_positions = verts[['x', 'y', 'z']].values   # [39, 3] metres

support_nodes = verts[verts['attribute'] == 'support']['v_idx'].tolist()
load_nodes    = verts[verts['attribute'] == 'load']['v_idx'].tolist()

print(f"Geometry loaded:")
print(f"  Nodes: {len(verts)}, Members: {len(df_edges)}")
print(f"  Support nodes: {support_nodes}")
print(f"  Load nodes: {load_nodes}")
print()

df_slots, feasibility_mask, member_forces, stats = stage_feas.build_cost_filter(
    node_positions=node_positions,
    edges_df=df_edges,
    stock_df=df_input_stock,
    support_nodes=support_nodes,
    load_nodes=load_nodes,
)
# display summary statistics about the feasibility filter results
print("\n" + "="*70)
print("FEASIBILITY FILTER SUMMARY")
print("="*70)
for k, v in stats.items():
    if isinstance(v, list):
        print(f"  {k:<35} {v}")
    elif isinstance(v, float):
        print(f"  {k:<35} {v:.4f}")
    else:
        print(f"  {k:<35} {v}")

print()
print(f"Member forces:")
print(f"  Tension     (N > 0): {(member_forces >  1.0).sum()} members")
print(f"  Compression (N < 0): {(member_forces < -1.0).sum()} members")
print(f"  Near-zero:           {(np.abs(member_forces) <= 1.0).sum()} members")
print(f"  Max tension:         {member_forces.max()/1000:.2f} kN")
print(f"  Max compression:     {member_forces.min()/1000:.2f} kN")

print("\nâœ“ Feasibility mask ready for cost matrix filtering")

In [ ]:
# STEP 1: COST MATRIX STAGE
cost_matrix, stock_prepared, logs = stage_cost.build_cost_matrix(
    df_slots=df_slots,
    df_input_stock=df_input_stock,
    feasibility_mask=feasibility_mask,
)

cost_matrix_df = pd.DataFrame(cost_matrix)
cost_matrix_df.to_csv(config.EXPORT_PATH / "c26_cost_matrix.csv")

# STEP 2: MILP STAGE
print("Starting MILP optimizer...")
milp_out = stage_milp.run_milp_stage(
    cost_matrix=cost_matrix,
    enriched_stock=stock_prepared,
    df_slots=df_slots,
    reclaimed_marker="RS",
    new_marker="NS",
    new_stock_max_uses=100,
    solver_msg=False,
    raise_on_infeasible_slots=False,
    )

status = milp_out.get("status")
df_results = milp_out.get("df_results", pd.DataFrame())
total_cost = milp_out.get("total_cost", float("inf"))
milp_summary = milp_out.get("summary", {})

df_results.to_csv(config.EXPORT_PATH / "c27_milp_results.csv", index=False)

print(
    f"MILP setup: {milp_summary.get('reclaimed_items', 0)} reclaimed + "
    f"{milp_summary.get('new_items', 0)} new stock items for {milp_summary.get('slots', len(df_slots))} slots"
)
print(f"MILP status: {status}")
print(f"Total assignment cost: {total_cost:.2f}")
if status != "Optimal":
    print("MILP did not solve to an optimal assignment for this geometry and stock set.")
if len(df_results) > 0:
    display(df_results.head(10))

In [ ]:
import c28_stage_GNN as stage_gnn
from c21_surrogate_io import load_surrogate_bundle

model_bundle = None
milp_status = milp_out.get("status")
milp_assignment = milp_out.get("milp_assignment")

if milp_status != "Optimal" or milp_assignment is None:
    print(f"Skipping GNN stage because MILP status is {milp_status!r}.")
elif MODEL_PREFIX:
    print(f"Loading surrogate model bundle with prefix: {MODEL_PREFIX}")
    model_bundle = load_surrogate_bundle(prefix_sm=MODEL_PREFIX)
else:
    model_bundle = globals().get("model_bundle")
    if model_bundle is None:
        print("No MODEL_PREFIX set and no model_bundle loaded; skipping GNN stage.")

if model_bundle is None:
    gnn_out = {
        "feasibility_score": 1.0,
        "structural_penalty": 0.0,
        "unsafe_member_ids": [],
    }
else:
    gnn_out = stage_gnn.run_gnn_stage(
        node_positions=node_positions,
        milp_assignment=milp_assignment,
        df_input_stock=df_input_stock,
        model_bundle=model_bundle,
    )

feasibility_score = float(gnn_out.get("feasibility_score", 1.0))
structural_penalty = float(gnn_out.get("structural_penalty", 0.0))
unsafe_member_ids = gnn_out.get("unsafe_member_ids", [])

print(f"\nGNN Feasibility Results:")
print(f"  Feasibility score:  {feasibility_score:.4f}  (1.0 = all members predicted safe)")
print(f"  Unsafe members:     {len(unsafe_member_ids)} / {stage_gnn.NUM_EDGES_PHYSICAL}")
print(f"  Structural penalty: {structural_penalty:.4f}  (w_structural=0.3)")
if unsafe_member_ids:
    print(f"  Unsafe member IDs:  {unsafe_member_ids[:20]}{'...' if len(unsafe_member_ids) > 20 else ''}")


In [ ]:
from workflows.c29_stage_fitness_score import run_fitness_stage
from workflows.c29_stage_normalization_bounds import run_normalization_bounds_stage

# ── Normalisation bounds ──────────────────────────────────────────────────────
bounds_out = run_normalization_bounds_stage(
    cost_matrix        = cost_matrix,
    df_logs            = logs,
    enriched_stock     = stock_prepared,
    df_slots           = df_slots,
    reclaimed_marker   = "RS",
    new_marker         = "NS",
    new_stock_max_uses = 100,
    solver_msg         = False,
    print_summary      = True,
)
normalization_constants = bounds_out["normalization_constants"]
print(normalization_constants)

# ── Fitness weights ───────────────────────────────────────────────────────────
# Single-shot evaluation — omega_4 fixed at w_start. The GA schedules it dynamically.
w_start = 0.2
w_end   = 0.8

fitness_weights = {
    "omega_1": 1.0,
    "omega_2": 1.0,
    "omega_3": 1.0,
    "omega_4": w_start,
}

# ── Fitness stage ─────────────────────────────────────────────────────────────
if df_results.empty:
    print("Skipping fitness stage — MILP returned no assignments.")
    fitness_result = {"objective": None, "is_feasible": False, "fitness": float("inf")}
    fitness_out = {"fitness_result": fitness_result, "normalization_constants": normalization_constants}
else:
    fitness_out = run_fitness_stage(
        df_results               = df_results,
        enriched_stock           = stock_prepared,
        df_slots                 = df_slots,
        total_cost               = total_cost,
        weight_config            = fitness_weights,
        normalization_constants  = normalization_constants,
        structural_infeasibility = 1.0 - feasibility_score,
        derive_normalization_constants = False,
        run_sanity_checks        = True,
        print_breakdown          = True,
    )
    fitness_result          = fitness_out["fitness_result"]
    normalization_constants = fitness_out["normalization_constants"]

fitness_export = dict(fitness_out)

print(f"\nFitness summary:")
print(f"  objective: {fitness_result.get('objective', 'n/a')}")
print(f"  feasible:  {fitness_result.get('is_feasible', 'n/a')}")
print(f"  fitness:   {fitness_result.get('fitness', 'n/a')}")

# ── Export ────────────────────────────────────────────────────────────────────
fitness_json_path = config.EXPORT_PATH / "c28_fitness_result.json"
fitness_csv_path  = config.EXPORT_PATH / "c28_fitness_result.csv"
fitness_json_path.parent.mkdir(parents=True, exist_ok=True)

def _to_builtin(value):
    return value.item() if hasattr(value, "item") else value

can_export = (not df_results.empty) and all(
    v == v and v not in (float("inf"), float("-inf"))
    for v in normalization_constants.values()
)

if can_export:
    fitness_export.pop("weights", None)
    fitness_export["weights"] = {k: float(v) for k, v in fitness_weights.items()}
    fitness_export["normalization_constants"] = {k: float(v) for k, v in normalization_constants.items()}
    with open(fitness_json_path, "w", encoding="utf-8") as f:
        json.dump(fitness_export, f, indent=2)
    fitness_row = {
        **{k: _to_builtin(v) for k, v in fitness_result.items()},
        **fitness_weights,
        **{k: float(v) for k, v in normalization_constants.items()},
    }
    pd.DataFrame([fitness_row]).to_csv(fitness_csv_path, index=False)
    print(f"\nExported: {fitness_json_path.name}, {fitness_csv_path.name}")
else:
    print("Skipping fitness export — pipeline infeasible or normalisation bounds unavailable.")


# 8. Export
Write vertices and edges (with MILP assignments) to disk for Grasshopper reconstruction. Skipped if MILP returned no assignment.


In [ ]:
EXPORT_PREFIX = "c29_optimum"

if any(name not in globals() for name in ["df_vertices", "df_edges", "df_results"]) or df_results.empty:
    print("Skipping structural export — no MILP assignment available.")
else:
    df_edges_export = pd.merge(
        df_edges,
        df_results[["edge_id", "assigned_timber", "CO2_Penalty"]],
        on="edge_id", how="left",
    )
    df_edges_export["assigned_timber"] = df_edges_export["assigned_timber"].fillna("UNASSIGNED")
    df_edges_export["CO2_Penalty"]     = df_edges_export["CO2_Penalty"].fillna(0)

    df_vertices.to_csv(config.EXPORT_PATH / f"{EXPORT_PREFIX}_vertices.csv", index=False)
    df_edges_export.to_csv(config.EXPORT_PATH / f"{EXPORT_PREFIX}_edges.csv", index=False)

    n_matched = int((df_edges_export["assigned_timber"] != "UNASSIGNED").sum())
    print(f"Exported: {len(df_vertices)} vertices, {len(df_edges_export)} edges ({n_matched} matched)")
